# MongoDB Advanced: How to Guarantee Data Consistency

## Enrollment Transactions

**Difficulty: Advanced | ~40 min | Requires Labs 1–4**

*Lab 5 of 7 in the MongoDB Mastery series.*

In this lab, you will learn how **multi-document transactions** guarantee that related writes either all succeed or all fail together — no partial state, no silent data corruption.

You will learn how to:
1. Connect to a MongoDB Atlas cluster
2. Populate `courses` and `enrollments` collections with small seat counts
3. See why non-atomic multi-document writes are dangerous
4. Fix the problem with a real multi-document transaction
5. Observe the abort path when a course is full
6. Use change streams to watch enrollments in real time
7. Export and restore collection data as a lightweight backup
8. Generate a formatted summary report

In [ ]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

This installs the MongoDB Python driver (`pymongo`) with `srv` and `tls` extras, `python-dotenv` for loading credentials, and `certifi` for up-to-date CA certificates.

### Step 1 — Connect to MongoDB

In [ ]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file one directory up
load_dotenv("../.env")
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collections
db = client["school_db"]
courses = db["courses"]
enrollments = db["enrollments"]

print("Connected to MongoDB Atlas")

Same connection pattern as Labs 1–4 — `load_dotenv` reads from the shared `.env` file, and `certifi.where()` provides trusted CA certificates for SSL.

### Step 2 — Populate Courses and Enrollments

In [ ]:
from datetime import datetime

courses.drop()
enrollments.drop()

# Insert 4 courses with deliberately small seat counts so we can
# realistically demonstrate a course filling up and a transaction aborting
course_records = [
    {"course_id": "CS101",   "title": "Introduction to Computer Science",
     "seats_total": 3, "seats_available": 3},
    {"course_id": "CS201",   "title": "Data Structures and Algorithms",
     "seats_total": 5, "seats_available": 5},
    {"course_id": "MATH101", "title": "Calculus I",
     "seats_total": 2, "seats_available": 2},
    {"course_id": "PHYS101", "title": "General Physics I",
     "seats_total": 4, "seats_available": 4},
]
courses.insert_many(course_records)
print(f"Inserted {len(course_records)} courses.")

# Enrollments starts empty — we will add enrollment documents in later steps
print(f"Enrollments: {enrollments.count_documents({})} documents.")

Each course tracks `seats_total` (maximum capacity) and `seats_available` (how many spots remain). The small counts — especially MATH101 with only 2 seats — let us demonstrate real aborts when a course fills up, rather than just describing the possibility hypothetically.

### Step 3 — The Problem: Non-Atomic Enrollment

In [ ]:
# Two separate operations: decrement seats, then insert the enrollment record
cid = "CS101"

courses.update_one(
    {"course_id": cid, "seats_available": {"$gt": 0}},
    {"$inc": {"seats_available": -1}}
)
print(f"Step 3a: decremented seats_available for {cid}")

enrollments.insert_one({
    "enrollment_id": "ENR999",
    "student_id": "STU999",
    "student_name": "Test Student",
    "course_id": cid,
    "enrolled_at": datetime.utcnow()
})
print(f"Step 3b: inserted enrollment document for STU999")

# Show the current state
course_after = courses.find_one({"course_id": cid}, {"_id": 0})
enr_count = enrollments.count_documents({"course_id": cid})
print(f"\nCurrent state of {cid}: seats_available = {course_after['seats_available']}, enrollments = {enr_count}")

This *happens* to work when nothing goes wrong — the seat decrement and the enrollment insert both succeed. The problem is that they are **two separate operations**. If the process crashes or an error occurs after the first `update_one` but before the `insert_one`, you get a seat taken with no enrollment record — or if the insert succeeds but the update fails, an enrollment exists for a seat that was never actually reserved. MongoDB guarantees that each *individual* write is atomic, but it cannot guarantee that two separate writes will both succeed or both fail. This is exactly the problem multi-document transactions are designed to solve.

### Step 4 — The Fix: Multi-Document Transaction

In [ ]:
# Reset CS101 back to full capacity for a clean demo
courses.update_one(
    {"course_id": "CS101"},
    {"$set": {"seats_available": 3}}
)
enrollments.delete_many({"course_id": "CS101"})

def enroll_student(session, cid, student_id, student_name):
    """Atomically decrement a seat and insert the enrollment record."""
    result = courses.find_one_and_update(
        {"course_id": cid, "seats_available": {"$gt": 0}},
        {"$inc": {"seats_available": -1}},
        session=session
    )
    if result is None:
        raise Exception(f"No seats available in {cid}")
    enrollments.insert_one({
        "enrollment_id": f"ENR{student_id[-3:]}",
        "student_id": student_id,
        "student_name": student_name,
        "course_id": cid,
        "enrolled_at": datetime.utcnow()
    }, session=session)

# Enroll 3 students into CS101 (which has 3 seats) using transactions
students_to_enroll = [
    ("STU001", "Alice Johnson"),
    ("STU002", "Bob Smith"),
    ("STU003", "Charlie Brown"),
]

for sid, sname in students_to_enroll:
    with client.start_session() as session:
        session.with_transaction(
            lambda s: enroll_student(s, "CS101", sid, sname)
        )
    print(f"Enrolled {sname} ({sid}) in CS101")

# Verify the state
cs101 = courses.find_one({"course_id": "CS101"}, {"_id": 0})
enr = list(enrollments.find({"course_id": "CS101"}, {"_id": 0, "enrollment_id": 0}))
print(f"\nCS101 seats_available: {cs101['seats_available']}")
print(f"CS101 enrollments ({len(enr)}):")
for e in enr:
    print(f"  {e['student_id']}: {e['student_name']}")

Each enrollment runs inside a **session** and a **transaction**. The `find_one_and_update` call checks that `seats_available > 0` and decrements it — and the `insert_one` call adds the enrollment record — both within the same transaction. If either operation fails, the entire transaction rolls back: no seat is taken without an enrollment, and no enrollment exists without a seat. The `find_one_and_update` with the `seats_available: {$gt: 0}` filter is the key — it makes the availability check and the decrement atomic together, so no two transactions can race past each other to claim the last seat.

### Step 5 — Abort Path: Enrolling Into a Full Course

In [ ]:
# CS101 now has 0 seats_available — try to enroll one more student
try:
    with client.start_session() as session:
        session.with_transaction(
            lambda s: enroll_student(s, "CS101", "STU999", "Overflow Student")
        )
    print("ERROR: This line should not print — enrollment should have failed.")
except Exception as e:
    print(f"Transaction aborted as expected: {e}")

# Verify nothing changed
cs101 = courses.find_one({"course_id": "CS101"}, {"_id": 0})
overflow = enrollments.find_one({"student_id": "STU999"})
print(f"\nAfter attempted overflow:")
print(f"  seats_available: {cs101['seats_available']}")
print(f"  STU999 enrollment exists: {overflow is not None}")

The `find_one_and_update` inside the transaction returned `None` because no document matched the filter `{"seats_available": {"$gt": 0}}` — the course is full. Our `enroll_student` function raises an exception, which causes `with_transaction` to **abort the entire transaction**. The result: `seats_available` stays at 0 (it did not go negative), and no enrollment document was created for STU999. The database is left in a perfectly consistent state, even though we attempted an invalid operation.

### Step 6 — Change Streams: Watching Enrollments in Real Time

In [ ]:
# Open a change stream on the enrollments collection
pipeline = []
change_stream = enrollments.watch(pipeline)

# Perform one more enrollment to generate a change event
with client.start_session() as session:
    session.with_transaction(
        lambda s: enroll_student(s, "CS201", "STU010", "Diana Prince")
    )

# Read the change event from the stream
event = change_stream.next()

print(f"Change stream event:")
print(f"  operationType: {event['operationType']}")
print(f"  fullDocument.student_id: {event['fullDocument']['student_id']}")
print(f"  fullDocument.student_name: {event['fullDocument']['student_name']}")
print(f"  fullDocument.course_id: {event['fullDocument']['course_id']}")

change_stream.close()

A **change stream** is a real-time feed of all write operations on a collection. `enrollments.watch()` opens the stream, and `next()` blocks until the next change arrives. The event tells you exactly what happened — `operationType` is `"insert"`, and `fullDocument` contains the newly inserted enrollment document. Change streams are the foundation for event-driven architectures: trigger a notification, sync to another system, or build a real-time dashboard — all driven by the database itself, not by polling.

### Step 7 — Lightweight Backup and Restore from Python

In [ ]:
import json

# --- Backup: export every document in both collections to a JSON file ---
backup = {
    "courses": [doc for doc in courses.find({}, {"_id": 0})],
    "enrollments": [doc for doc in enrollments.find({}, {"_id": 0})],
}

backup_path = "school_db_backup.json"
with open(backup_path, "w") as f:
    json.dump(backup, f, indent=2, default=str)

print(f"Backup saved to {backup_path}")
print(f"  courses: {len(backup['courses'])} documents")
print(f"  enrollments: {len(backup['enrollments'])} documents")

In [ ]:
# --- Restore: wipe both collections and reinsert from the JSON backup ---
courses.drop()
enrollments.drop()
print(f"Dropped collections. courses: {courses.count_documents({})}, enrollments: {enrollments.count_documents({})}")

with open(backup_path, "r") as f:
    restored = json.load(f)

if restored["courses"]:
    courses.insert_many(restored["courses"])
if restored["enrollments"]:
    enrollments.insert_many(restored["enrollments"])

print(f"Restored. courses: {courses.count_documents({})}, enrollments: {enrollments.count_documents({})}")

This is a lightweight backup/restore pattern you can do entirely from Python — exporting documents as JSON and reinserting them to restore state. For production databases, use MongoDB Atlas's built-in `mongodump` / `mongorestore` CLI tools or Atlas's automated backup and point-in-time recovery, which operate at the storage engine level and are far more robust. The Python approach above is useful for small datasets, testing, or when you need to snapshot a specific subset of data without CLI access.

### Step 8 — Summary Report

In [ ]:
total_courses = courses.count_documents({})
total_enrollments = enrollments.count_documents({})

print("       ENROLLMENT TRANSACTIONS")
print(f"\nTotal courses: {total_courses}")
print(f"Total enrollments: {total_enrollments}")

print("\nSeats remaining per course:")
for c in courses.find({}, {"_id": 0}).sort("course_id", 1):
    enr_count = enrollments.count_documents({"course_id": c["course_id"]})
    print(f"  {c['course_id']}: {c['seats_available']}/{c['seats_total']} seats available ({enr_count} enrolled)")

Collects the key metrics from both collections into one formatted report — course counts, remaining seats per course, and total enrollments across the system.